# UP Contingent Planning Tutorial

<a target="_blank" href="https://colab.research.google.com/github/aiplan4eu/up-cpor/blob/master/notebooks/CPOR_and_SDR_running_examples.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Contingent planning under partial observability and sensing actions is an importat problem in automated planning.
This notebook provides examples on using the contingent planning package of the unified planning framework. The package supports offline planning, where a complete plan graph is constructed, and online planning, where the planner interacts with the environment during execution, receving observations and computing which action to perfrom next.

For information about contingent planning, and the algorithms used here can be found at:


*   Shlomi Maliah, Radimir Komarnitsky, Guy Shani: Computing Contingent Plan Graphs using Online Planning. JAAMAS 16(1): 1:1-1:30 (2021)
*   Ronen I. Brafman, Guy Shani: Replanning in Domains with Partial Information and Sensing Actions. J. Artif. Intell. Res. 45: 565-600 (2012)

For questions or comments please contact Guy Shani - shanigu@bgu.ac.il.


## If you would like that the solution will be print change the parameter to True

In [1]:
SOLUTION_PRINTED = False

def print_CPOR_sol(p_planNode, tab = ""):
    if p_planNode is not None:
        x = p_planNode
        print(tab, x.action_instance)
        for c in x.children:
            print_CPOR_sol(c[1], tab+"\t")

## Install

 We begin by installing the latest version of the up-cpor package.

In [ ]:
!pip install git+https://github.com/aiplan4eu/up-cpor

## Problem Setup

We can load any contingent planning problem supported by the unified-planning framework. Below is a simple 2-blocks blocksworld prblem:

In [2]:
DOMAIN = """
(define (domain blocksworld)
(:requirements :contingent)
(:types block)
(:predicates (clear ?x - block)
             (on-table ?x - block)
             (on ?x ?y - block)
             (same ?x ?y - block))



  (:action senseON
   :parameters (?b1 ?b2 - block)
   :observe (on ?b1 ?b2))

  (:action senseCLEAR
   :parameters (?b1 - block)
   :observe (clear ?b1))

  (:action senseONTABLE	
   :parameters (?b1 - block)
   :observe (on-table ?b1))

(:action move-b-to-b
  :parameters (?bm ?bf ?bt - block)
  :precondition (and (clear ?bm) (clear ?bt) (on ?bm ?bf) (not (same ?bm ?bt)))
  :effect (and (not (clear ?bt)) (not (on ?bm ?bf))
               (on ?bm ?bt) (clear ?bf)))

(:action move-to-t
  :parameters (?b ?bf - block)
  :precondition (and (clear ?b) (on ?b ?bf))
  :effect (and (on-table ?b) (not (on ?b ?bf)) (clear ?bf)))

(:action move-t-to-b
  :parameters (?bm ?bt - block)
  :precondition (and (clear ?bm) (clear ?bt) (on-table ?bm) (not (same ?bm ?bt)))
  :effect (and (not (clear ?bt)) (not (on-table ?bm))
               (on ?bm ?bt))))
"""

PROBLEM = """
(define (problem BW-rand-3)
(:domain blocksworld)
(:objects b1 b2 - block)
(:init
(on-table b1)
(clear b2)
(same b1 b1)
(same b2 b2)


(unknown (on b2 b1))
(unknown (on-table b2))
(unknown (clear b1))


(oneof
(on-table b2)
(on b2 b1)
)
(oneof
(clear b1)
(on b2 b1)
)
)
(:goal
(and
(on b1 b2)
)
)
)
"""

In [3]:
from unified_planning.io import PDDLReader

# Creating a PDDL reader
reader = PDDLReader()

problem = reader.parse_problem_string(DOMAIN, PROBLEM)
problem

problem name = bw-rand-3

types = [block]

fluents = [
  bool clear[x=block]
  bool on-table[x=block]
  bool on[x=block, y=block]
  bool same[x=block, y=block]
]

actions = [
  sensing-action senseon(block b1, block b2) {
    preconditions = [
    ]
    effects = [
    ]
    observations = [
      on(b1, b2)
    ]
  }
  sensing-action senseclear(block b1) {
    preconditions = [
    ]
    effects = [
    ]
    observations = [
      clear(b1)
    ]
  }
  sensing-action senseontable(block b1) {
    preconditions = [
    ]
    effects = [
    ]
    observations = [
      on-table(b1)
    ]
  }
  action move-b-to-b(block bm, block bf, block bt) {
    preconditions = [
      (clear(bm) and clear(bt) and on(bm, bf) and (not same(bm, bt)))
    ]
    effects = [
      clear(bt) := false
      on(bm, bf) := false
      on(bm, bt) := true
      clear(bf) := true
    ]
  }
  action move-to-t(block b, block bf) {
    preconditions = [
      (clear(b) and on(b, bf))
    ]
    effects = [
      on-

# Offline Planning Example

We now deonstrate how to compute a complete plan graph for a contingent problem, where nodes are labeled by actions, and edges are labeled by observations. The package currently implements only the CPOR offline planner. We initialize the planner, and then call the solve method to compute a solution.

After a solution plan tree is computed, we can save the resulting plan to a file.

In [4]:
import os

import unified_planning.environment as environment
from unified_planning.shortcuts import OneshotPlanner
from unified_planning.engines.results import PlanGenerationResultStatus


env = environment.get_environment()
env.factory.add_engine('CPORPlanning', 'up_cpor.engine', 'CPORImpl')

with OneshotPlanner(name='CPORPlanning') as planner:
    result = planner.solve(problem)
    if SOLUTION_PRINTED:
        print_CPOR_sol(result.plan.root_node)
    if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
        print(f'{planner.name} found a valid plan!')
        print(f'Success')
    else:
        print('No plan found!')

NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 11 of `/tmp/ipykernel_96055/556732106.py`, you are using the following planning engine:
  * Engine name: CPOR
  * Developers:  Guy Shani
  * Description: CPOR is an offline contingent planner.
  *               It computes a complete plan tree (or graph) where each node is labeled by an action, and edges are labeled by observations.
  *              The leaves of the plan tree correspond to goal states.

Started offline planning for bw-rand-3, 3/12/2026 2:36:05 PM


Cueing down from goal distance: 2 into depth [1]
                                1            [1]
                                0            0:1:

time spent: instantiating 58 easy, 64 hard action templates
           reachability analysis, yielding 71 facts and 90 actions
             creating final representation with 65 

# Using a UP Classical Planner Inside CPOR

CPOR (and SDR) operate by creating classical planning problems that model the partial knowledge, and solve them, to obtain a heuristic about which action to choose next. The CPOR package contains an internal impementation of the popular FF classical planner, by Joerg Hoffman. However, the package supports running any UP classical solver. We demonstrate here how the UP implementation of Tamer can be used instead of the internal FF.

In [5]:
!pip install up-tamer

In [6]:
import unified_planning.environment as environment
from unified_planning.engines.results import PlanGenerationResultStatus
from unified_planning.shortcuts import OneshotPlanner

env = environment.get_environment()
env.factory.add_meta_engine('MetaCPORPlanning', 'up_cpor.engine', 'CPORMetaEngineImpl')

with OneshotPlanner(name='MetaCPORPlanning[tamer]') as planner:
    result = planner.solve(problem)
    if SOLUTION_PRINTED:
        print_CPOR_sol(result.plan.root_node)
    if result.status == PlanGenerationResultStatus.SOLVED_SATISFICING:
        print(f'{planner.name} found a valid plan!')
        print(f'Success')
    else:
        print('No plan found!')

  *** Credits ***
  * In operation mode `OneshotPlanner` at line 8 of `/tmp/ipykernel_96055/3508204899.py`, you are using the following planning engine:
  * Engine name: Conitngent Planning Algorithms
  * Developers:  Guy Shani
  * Description: Algorithms for offline and online decision making under partial observability and sensing actions

Started offline planning for bw-rand-3, 3/12/2026 2:36:10 PM


Cueing down from goal distance: 2 into depth [1]
                                1            [1]
                                0            0:1:

time spent: instantiating 58 easy, 64 hard action templates
           reachability analysis, yielding 71 facts and 90 actions
             creating final representation with 65 relevant facts
             building connectivity graph
            searching, evaluating 3 states, to a max depth of 1



Replanning: 0, goal leaves: 0 closed states: 0 elpased time: 0

Cueing down from goal distance: 2 into depth [1]
                              

# Online Contingent Planning

While in offline planning the planner computes a complete plan graph, in online planning we take a closed loop approach, where an agent interacts with the environment during execution.

The agent executes an action in the environment, and then receives an observation as a result in the action. In goal-based contingent planning this loop continues until the agent ensures that the goal has been achieved.

The CPOR package implements the SDR contingent (re)planner. SDR operates by translating the contingent problem into a classical problem, solving it using a classical solver, and then executing the resulting actions, if their preconditions hold. When an unexpected observation was received, SDR replans.

The code below demonstrates how SDR can be used, interacting with a simulated environment, which is also implemented inside the CPOR package. The while loop below implements the closed loop process.



In [7]:
import unified_planning.environment as environment
from unified_planning.shortcuts import ActionSelector
from up_cpor.simulator import SDRSimulator

env = environment.get_environment()
env.factory.add_engine('SDRPlanning', 'up_cpor.engine', 'SDRImpl')

with ActionSelector(name='SDRPlanning', problem=problem) as solver:
    simulatedEnv = SDRSimulator(problem)
    while not simulatedEnv.is_goal_reached():
        action = solver.get_action()
        observation = simulatedEnv.apply(action)
        solver.update(observation)
        if SOLUTION_PRINTED:
            print(f"Action: {action}")
        if observation:
            print(f"Observation: {observation}")
    print(f'{solver.name} found a valid plan!')
    print(f'Success')

  *** Credits ***
  * In operation mode `ActionSelector` at line 8 of `/tmp/ipykernel_96055/1845009071.py`, you are using the following planning engine:
  * Engine name: SDR
  * Developers:  Guy Shani
  * Description: SDR is an online contingent replanner.
  *              It provides one action at a time, and then awaits to receive an observation from the environment.



Cueing down from goal distance: 3 into depth [1]
                                2            [1]
                                1            [1]
                                0            0:1:2:

time spent: instantiating 58 easy, 64 hard action templates
           reachability analysis, yielding 75 facts and 90 actions
             creating final representation with 65 relevant facts
             building connectivity graph
            searching, evaluating 4 states, to a max depth of 1



Observation: {on(b2, b1): true}


Cueing down from goal distance: 2 into depth [1]
                                1        